In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import numpy as np
import pandas as pd
from sklearn.preprocessing import RobustScaler
from tqdm import tqdm
import os
import random
import matplotlib.pyplot as plt
import neptune as neptune
from neptune.types import File
from dotenv import load_dotenv

In [ ]:
load_dotenv("../.env")

PATH = os.getenv("btc")

In [ ]:
data = pd.read_csv(PATH)

In [ ]:
device = "cuda"

In [ ]:
LR = 0.0001
L2_DECAY = 5e-5
DROPOUT_RATE = 0.4
BATCH_SIZE = 512
HIDDEN_SIZE = 384
NUM_LAYERS = 3
EPOCHS = 100
PATIENCE = 5
WINDOW_SIZE = 30
QUANTILE_CONFIG = [0.1, 0.5, 0.9]
TARGET_COLUMN = 'Return'

In [ ]:
num_workers = 0

In [ ]:
NEPTUNE_PROJECT = "isaakclarke/aitrain"
NEPTUNE_API_TOKEN = os.getenv("neptune_api_token")

In [ ]:
PARAMS = {
    'learning_rate': LR,
    'l2_decay': L2_DECAY,
    'dropout_rate': DROPOUT_RATE,
    'batch_size': BATCH_SIZE,
    'hidden_size': HIDDEN_SIZE,
    'num_layers': NUM_LAYERS,
    'window_size': WINDOW_SIZE,
    'quantiles': QUANTILE_CONFIG,
    'epochs': EPOCHS,
    'patience': PATIENCE,
    'target_column': TARGET_COLUMN,
    'model_type': 'GRU_Quantile',
    'optimizer': 'Adam',
    'loss_function': 'Pinball Loss'
}

In [ ]:
def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False
set_seed()

In [ ]:
class CleanTimeseriesDataset(Dataset):
    def __init__(self, X_data, Y_data, window_size):
        assert len(X_data) == len(Y_data), "X and Y data must have the same length"
        
        self.X_data = torch.tensor(X_data.values, dtype=torch.float32)
        self.Y_data = torch.tensor(Y_data.values, dtype=torch.float32) 
        
        self.window_size = window_size
        self.len = len(self.X_data) - self.window_size

    def __len__(self):
        return self.len

    def __getitem__(self, idx):
        X = self.X_data[idx : idx + self.window_size]
        
        Y = self.Y_data[idx + self.window_size]
        
        return X, Y.squeeze(0)

In [ ]:
class PinballLoss(nn.Module):
    def __init__(self, quantiles):
        super().__init__()
        self.quantiles = torch.tensor(quantiles, dtype=torch.float32, device=device).unsqueeze(0).unsqueeze(0)

    def forward(self, predictions, target):
        target = target.unsqueeze(-1) 
        
        errors = target - predictions.unsqueeze(1) 
        
        loss = torch.max(self.quantiles * errors, (self.quantiles - 1) * errors)
        
        return torch.mean(torch.sum(loss, dim=-1))

In [ ]:
class QuantileGRUModel(nn.Module):
    def __init__(self, input_size, hidden_size, num_layers, quantiles, dropout_rate):
        super(QuantileGRUModel, self).__init__()
        
        self.gru = nn.GRU(
            input_size, 
            hidden_size, 
            num_layers, 
            batch_first=True,
            dropout=dropout_rate
        )
        self.dropout = nn.Dropout(dropout_rate)
        self.fc = nn.Linear(hidden_size, 1 * len(quantiles))
        
    def forward(self, x):
        _, h_n = self.gru(x) 
        out = h_n[-1] 
        out = self.dropout(out)
        out = self.fc(out)
        return out.view(-1, len(QUANTILE_CONFIG))

In [ ]:
def feature_engineering(df: pd.DataFrame):
    df[TARGET_COLUMN] = df['Close'].pct_change()
    df = df.dropna().reset_index(drop=True) 

    df['EMA_12'] = df['Close'].ewm(span=12, adjust=False).mean()
    df['EMA_26'] = df['Close'].ewm(span=26, adjust=False).mean()
    df['MACD'] = df['EMA_12'] - df['EMA_26']
    df['MACD_Signal'] = df['MACD'].ewm(span=9, adjust=False).mean()
    df['MACD_Hist'] = df['MACD'] - df['MACD_Signal']

    def calculate_rsi(data, window=14):
        delta = data['Close'].diff()
        gain = delta.where(delta > 0, 0)
        loss = -delta.where(delta < 0, 0)
        avg_gain = gain.ewm(com=window-1, min_periods=window).mean()
        avg_loss = loss.ewm(com=window-1, min_periods=window).mean()
        rs = avg_gain / avg_loss
        data['RSI'] = 100 - (100 / (1 + rs))
        return data
    df = calculate_rsi(df)

    df['Return_Lag_1'] = df[TARGET_COLUMN].shift(1)
    df['Return_Lag_5'] = df[TARGET_COLUMN].shift(5)
    df['Cum_Return_5'] = df[TARGET_COLUMN].rolling(window=5).sum()
    df['Cum_Return_20'] = df[TARGET_COLUMN].rolling(window=20).sum()

    df['Vol_5_min'] = df[TARGET_COLUMN].rolling(window=5).std()    
    df['Vol_20_min'] = df[TARGET_COLUMN].rolling(window=20).std()   
    df['High_Low_Range'] = (df['High'] - df['Low']) / df['Open'] 
    df['TR'] = np.maximum.reduce([df['High'] - df['Low'], np.abs(df['High'] - df['Close'].shift(1)), np.abs(df['Low'] - df['Close'].shift(1))])
    df['ATR'] = df['TR'].rolling(window=14).mean()

    df['Log_Volume'] = np.log(df['Volume'] + 1e-6)
    df['Vol_SMA_20'] = df['Volume'].rolling(window=20).mean()
    df['Dist_SMA_20'] = (df['Close'] - df['Close'].rolling(window=20).mean()) / df['Close']
    df['Close_Pos'] = (df['Close'] - df['Low']) / (df['High'] - df['Low'])

    df = df.dropna().reset_index(drop=True)

    X_cols = [
        'MACD', 'MACD_Signal', 'MACD_Hist', 'RSI', 'Return_Lag_1', 'Return_Lag_5', 'Cum_Return_5', 'Cum_Return_20',
        'Vol_5_min', 'Vol_20_min', 'High_Low_Range', 'ATR', 'Log_Volume', 'Vol_SMA_20', 'Dist_SMA_20', 'Close_Pos'
    ]
    
    return df, X_cols

In [ ]:
data_processed, X_cols = feature_engineering(data)

In [ ]:
train_split = int(len(data_processed) * 0.9)
data_train = data_processed.iloc[:train_split].copy().reset_index(drop=True)
data_test = data_processed.iloc[train_split:].copy().reset_index(drop=True)

scaler_X = RobustScaler()
scaler_Y = RobustScaler()

X_train = data_train[X_cols]
Y_train = data_train[[TARGET_COLUMN]] 

X_train_scaled = scaler_X.fit_transform(X_train)
Y_train_scaled = scaler_Y.fit_transform(Y_train)

X_test_scaled = scaler_X.transform(data_test[X_cols])
Y_test_scaled = scaler_Y.transform(data_test[[TARGET_COLUMN]])

X_train_scaled = pd.DataFrame(X_train_scaled, columns=X_cols)
Y_train_scaled = pd.DataFrame(Y_train_scaled, columns=[TARGET_COLUMN])
X_test_scaled = pd.DataFrame(X_test_scaled, columns=X_cols)
Y_test_scaled = pd.DataFrame(Y_test_scaled, columns=[TARGET_COLUMN])

In [ ]:
X_train_scaled.shape

In [ ]:
Y_train_scaled.shape

In [ ]:
train_dataset = CleanTimeseriesDataset(X_train_scaled, Y_train_scaled, WINDOW_SIZE)
test_dataset = CleanTimeseriesDataset(X_test_scaled, Y_test_scaled, WINDOW_SIZE)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, drop_last=True, num_workers=num_workers)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False, drop_last=True, num_workers=num_workers)

In [ ]:
input_size = X_train_scaled.shape[1] 

model = QuantileGRUModel(
    input_size=input_size, hidden_size=HIDDEN_SIZE, num_layers=NUM_LAYERS, 
    quantiles=QUANTILE_CONFIG, dropout_rate=DROPOUT_RATE
).to(device)

criterion = PinballLoss(quantiles=QUANTILE_CONFIG)
optimizer = optim.Adam(model.parameters(), lr=LR, weight_decay=L2_DECAY)

In [ ]:
try:
    run = neptune.init_run( 
        project=NEPTUNE_PROJECT, 
        api_token=NEPTUNE_API_TOKEN,
        tags=["GRU", "Quantile", f"Layers_{NUM_LAYERS}", f"HSize_{HIDDEN_SIZE}"]
    )
    run["parameters"] = PARAMS
    run["feature_columns"] = X_cols
    print("-> Neptune online.")
except Exception as e:
    run = None
    print(f"-> error init Neptune, logging off: {e}")

In [ ]:
best_val_loss = float('inf')
patience_counter = 0
train_losses, val_losses = [], []

for epoch in range(1, EPOCHS + 1):
    
    model.train()
    total_train_loss = 0.0
    for inputs, targets in tqdm(train_loader, desc=f"Epoch {epoch} Train"):
        inputs = inputs.to(device)
        targets = targets.to(device)
        optimizer.zero_grad()
        predictions = model(inputs)
        loss = criterion(predictions, targets) 
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()
        total_train_loss += loss.item() * inputs.size(0)
        if run: run["batch_metrics/train_loss"].log(loss.item())

    epoch_train_loss = total_train_loss / len(train_dataset)
    train_losses.append(epoch_train_loss)
    
    model.eval()
    total_val_loss = 0.0
    with torch.no_grad():
        for inputs, targets in tqdm(test_loader, desc=f"Epoch {epoch} Val"):
            inputs = inputs.to(device)
            targets = targets.to(device)
            predictions = model(inputs)
            loss = criterion(predictions, targets)
            total_val_loss += loss.item() * inputs.size(0)
            if run: run["batch_metrics/val_loss"].log(loss.item())

    epoch_val_loss = total_val_loss / len(test_dataset)
    val_losses.append(epoch_val_loss)

    if run:
        run["metrics/train_loss"].log(epoch_train_loss)
        run["metrics/val_loss"].log(epoch_val_loss)

    print(f"Epoch {epoch}/{EPOCHS}: Train Loss: {epoch_train_loss:.6f}, Val Loss: {epoch_val_loss:.6f}")
    
    if epoch_val_loss < best_val_loss:
        best_val_loss = epoch_val_loss
        patience_counter = 0
        filepath = './best_pytorch_quantile_model.pth'
        torch.save(model.state_dict(), filepath)
        print("-> model save")
        if run: run["model_checkpoints/best_model"].upload(File(filepath))
    else:
        patience_counter += 1
        if patience_counter >= PATIENCE:
            print(f"--- Early Stopping epoch {epoch} ---")
            break

if run: 
    run.stop()
    print("\n-> Neptune offline")

In [ ]:
plt.figure(figsize=(12, 6))
plt.plot(train_losses, label='Train Pinball Loss', color='blue', alpha=0.7)
plt.plot(val_losses, label='Validation Pinball Loss', color='orange', alpha=0.7)
plt.title('Кривые обучения')
plt.xlabel('Эпоха')
plt.ylabel('Pinball Loss')
plt.legend()
plt.grid(True, which='both', linestyle='--', linewidth=0.5)
plt.show()

def visualize_predictions(model, test_loader, scaler_Y, device, num_samples=500):

    model.eval()
    all_targets = []
    all_preds = []
    
    with torch.no_grad():
        for inputs, targets in test_loader:
            inputs = inputs.to(device)
            preds = model(inputs)
            
            all_preds.append(preds.cpu().numpy())
            all_targets.append(targets.cpu().numpy())
            
            if len(all_targets) * test_loader.batch_size > num_samples:
                break
    
    predictions_scaled = np.concatenate(all_preds, axis=0)[:num_samples]
    targets_scaled = np.concatenate(all_targets, axis=0)[:num_samples]

    targets = scaler_Y.inverse_transform(targets_scaled.reshape(-1, 1)).flatten()
    
    q10 = scaler_Y.inverse_transform(predictions_scaled[:, 0].reshape(-1, 1)).flatten()
    q50 = scaler_Y.inverse_transform(predictions_scaled[:, 1].reshape(-1, 1)).flatten()
    q90 = scaler_Y.inverse_transform(predictions_scaled[:, 2].reshape(-1, 1)).flatten()
    
    
    plt.figure(figsize=(18, 8))
    
    plt.plot(targets, label='Реальная Доходность', color='black', linewidth=1, alpha=0.8)
    plt.plot(q50, label='Медиана Прогноза (q50)', color='red', linestyle='--', alpha=0.7)
    plt.fill_between(range(len(targets)), q10, q90, color='green', alpha=0.2, label='Интервал Уверенности (10-90%)')
    
    plt.title(f'Прогноз Доходности на тестовом наборе ({num_samples} точек)')
    plt.xlabel('Временной шаг')
    plt.ylabel('Доходность (Return)')
    plt.legend()
    plt.grid(True, which='both', linestyle='--', linewidth=0.5)
    plt.show()

visualize_predictions(model, test_loader, scaler_Y, device, num_samples=500)